# Filtered GRU Remote Suite

Colab에서 `final_dataset_filtered.csv` / `final_dataset_3class_filtered.csv` 기반 GRU 후보를 버전별로 실행한다.

- `v30_binary_segment`: `label`, positive=`1`, `segment_max`
- `v31_binary_last`: `label`, positive=`1`, `last_frame`
- `v32_3class_fall_any`: `label_3class`, positive=`1,2`, `segment_max`
- `v33_3class_falling_only`: `label_3class`, positive=`1`, `last_frame`

각 버전은 같은 모델 후보(`gru_64_32`, `gru_96_48`, `gru_128_64`, `gru_64_32_light`)와 같은 시각화/STM32 export 파이프라인을 사용한다.

In [ ]:
# Runtime setup
from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import json
import subprocess
from pathlib import Path
from datetime import datetime

REPO_URL = 'https://github.com/2025-fin-graduate-project/Falling-Detection-Development.git'
BRANCH = 'codex/filtered-gru-models'
PROJECT_ROOT = Path('/content/drive/MyDrive/Graduate-Project/Falling-Model-Development')

PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
if not (PROJECT_ROOT / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_ROOT)], check=True)

os.chdir(PROJECT_ROOT)
subprocess.run(['git', 'fetch', 'origin'], check=True)
subprocess.run(['git', 'checkout', BRANCH], check=True)
subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], check=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('BRANCH =', BRANCH)
print('timestamp =', datetime.now().isoformat(timespec='seconds'))

In [ ]:
# Dependencies
# Colab already includes most packages, but pin nothing here so the notebook remains portable.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'matplotlib', 'scikit-learn', 'pandas', 'numpy'
], check=True)

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

In [ ]:
# Experiment controls
RUN_VERSIONS = ['all']  # examples: ['v30_binary_segment'], ['v30_binary_segment', 'v31_binary_last'], ['all']
MODELS = 'all'          # or 'gru_64_32,gru_96_48'
EPOCHS = 40
BATCH_SIZE = 64
MIN_VAL_RECALL = 0.86
QUANT_EVAL_MAX_WINDOWS = 5000
SKIP_STM32_EXPORT = False

DATASET_DIR = PROJECT_ROOT / 'dataset'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts' / 'filtered_gru_remote_suite'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENTS = {
    'v30_binary_segment': {
        'csv': DATASET_DIR / 'final_dataset_filtered.csv',
        'source_csv': DATASET_DIR / 'final_dataset.csv',
        'label_column': 'label',
        'positive_labels': '1',
        'label_mode': 'segment_max',
        'description': '2-class label, segment_max window label',
    },
    'v31_binary_last': {
        'csv': DATASET_DIR / 'final_dataset_filtered.csv',
        'source_csv': DATASET_DIR / 'final_dataset.csv',
        'label_column': 'label',
        'positive_labels': '1',
        'label_mode': 'last_frame',
        'description': '2-class label, last_frame window label',
    },
    'v32_3class_fall_any': {
        'csv': DATASET_DIR / 'final_dataset_3class_filtered.csv',
        'source_csv': DATASET_DIR / 'final_dataset_3class.csv',
        'label_column': 'label_3class',
        'positive_labels': '1,2',
        'label_mode': 'segment_max',
        'description': '3-class labels mapped falling+fallen to binary fall',
    },
    'v33_3class_falling_only': {
        'csv': DATASET_DIR / 'final_dataset_3class_filtered.csv',
        'source_csv': DATASET_DIR / 'final_dataset_3class.csv',
        'label_column': 'label_3class',
        'positive_labels': '1',
        'label_mode': 'last_frame',
        'description': '3-class labels mapped only falling to binary fall',
    },
}

selected_versions = list(EXPERIMENTS) if 'all' in RUN_VERSIONS else RUN_VERSIONS
display(pd.DataFrame([
    {'version': key, **{k: str(v) for k, v in cfg.items()}}
    for key, cfg in EXPERIMENTS.items()
    if key in selected_versions
]))

In [ ]:
# Build filtered datasets if missing
def ensure_filtered_dataset(target_csv: Path, source_csv: Path):
    if target_csv.exists():
        print(f'[OK] filtered dataset exists: {target_csv}')
        return
    if not source_csv.exists():
        raise FileNotFoundError(f'Missing source dataset: {source_csv}')
    print(f'[BUILD] {source_csv.name} -> {target_csv.name}')
    subprocess.run([
        sys.executable, 'scripts/build_filtered_dataset.py',
        '--input', str(source_csv),
        '--output', str(target_csv),
        '--min-cutoff', '0.5',
        '--beta', '0.3',
        '--conf-thr', '0.15',
        '--ema-deriv-alpha', '0.4',
    ], check=True)

for version in selected_versions:
    cfg = EXPERIMENTS[version]
    ensure_filtered_dataset(cfg['csv'], cfg['source_csv'])

In [ ]:
# Run remote training jobs sequentially
run_records = []

for version in selected_versions:
    cfg = EXPERIMENTS[version]
    output_dir = ARTIFACT_DIR / version
    cmd = [
        sys.executable, 'scripts/train_filtered_gru_suite.py',
        '--csv-path', str(cfg['csv']),
        '--output-dir', str(output_dir),
        '--label-column', cfg['label_column'],
        '--positive-labels', cfg['positive_labels'],
        '--label-mode', cfg['label_mode'],
        '--models', MODELS,
        '--epochs', str(EPOCHS),
        '--batch-size', str(BATCH_SIZE),
        '--min-val-recall', str(MIN_VAL_RECALL),
        '--quant-eval-max-windows', str(QUANT_EVAL_MAX_WINDOWS),
    ]
    if SKIP_STM32_EXPORT:
        cmd.append('--skip-stm32-export')

    print('\n' + '=' * 100)
    print(f'RUN {version}: {cfg["description"]}')
    print(' '.join(cmd))
    print('=' * 100)
    subprocess.run(cmd, check=True)
    run_records.append({'version': version, 'output_dir': str(output_dir), **cfg})

pd.DataFrame(run_records).to_csv(ARTIFACT_DIR / 'remote_run_records.csv', index=False)
display(pd.DataFrame(run_records))

In [ ]:
# Aggregate version/model comparisons
frames = []
for version in selected_versions:
    comparison_path = ARTIFACT_DIR / version / 'model_comparison.csv'
    if comparison_path.exists():
        df = pd.read_csv(comparison_path)
        df.insert(0, 'version', version)
        frames.append(df)

if not frames:
    raise FileNotFoundError('No model_comparison.csv files found.')

summary = pd.concat(frames, ignore_index=True)
summary_path = ARTIFACT_DIR / 'all_versions_model_comparison.csv'
summary.to_csv(summary_path, index=False)

metric_cols = [c for c in ['test_accuracy', 'test_precision', 'test_recall', 'test_macro_f1', 'int8_accuracy', 'int8_macro_f1'] if c in summary.columns]
display(summary[['version', 'model', 'threshold', 'min_consecutive'] + metric_cols].sort_values('test_macro_f1', ascending=False))

plot_df = summary.copy()
plot_df['version_model'] = plot_df['version'] + '\n' + plot_df['model']
ax = plot_df.set_index('version_model')[['test_accuracy', 'test_macro_f1']].plot(kind='bar', figsize=(16, 5))
ax.axhline(0.90, color='crimson', linestyle='--', linewidth=1.2)
ax.set_ylim(0, 1)
ax.set_ylabel('Score')
ax.set_title('Filtered GRU remote suite summary')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
summary_png = ARTIFACT_DIR / 'all_versions_summary.png'
plt.savefig(summary_png, dpi=160)
plt.show()

print('summary csv:', summary_path)
print('summary png:', summary_png)

In [ ]:
# Display generated comparison images for quick review
for version in selected_versions:
    image_path = ARTIFACT_DIR / version / 'model_comparison.png'
    if image_path.exists():
        print('\n', version, image_path)
        display(Image(filename=str(image_path)))